# Neural CDE-AFT pour la prédiction du temps avant l'IRT

Ce notebook met en œuvre un modèle **Neural Controlled Differential Equation (Neural CDE)** pour prédire le **temps restant avant l'initiation du traitement de suppléance rénale (IRT)** à partir des trajectoires cliniques longitudinales des patients.

L'architecture s'appuie sur l'implémentation de référence de **Kidger et al. (2020)** (JAX, Equinox et Diffrax) et l'adapte à un problème d'analyse de survie en utilisant une **tête AFT (Accelerated Failure Time)** basée sur une distribution log-normale.

Les trajectoires sont construites à partir des mesures répétées (âge, eGFR, hypertension, événement urologique, etc.), tandis que les variables statiques sont intégrées à l'état latent final avant la prédiction des paramètres de survie.

## Organisation du notebook

1. **Chargement et préparation des données**
2. **Construction des trajectoires contrôlées** à partir des visites longitudinales
3. **Définition du modèle Neural CDE** pour encoder la dynamique des patients
4. **Fusion de l'état latent avec les variables statiques**
5. **Tête de survie AFT** prédisisant les paramètres \((\mu, \sigma)\) de la loi log-normale
6. **Fonction de perte Log-Normal AFT** prenant en compte la censure
7. **Optimisation des hyperparamètres** avec Optuna
8. **Évaluation des performances** (C-index, MAE, RMSE et analyses par nombre de visites)

## Imports

In [ ]:
import math
import time
import collections

import diffrax
import equinox as eqx
import jax
import jax.nn as jnn
import jax.numpy as jnp
import jax.random as jr
import jax.scipy as jsp
import numpy as np
import pandas as pd
import optax
import optuna
import polars as pl
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from lifelines.utils import concordance_index

from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

optuna.logging.set_verbosity(optuna.logging.WARNING)


## Étape 1 — Chargement des données

In [ ]:
# =============================================================================
# Chargement des données longitudinales
#
# train_dynamic :
#   - utilisé pour l'entraînement du modèle ;
#   - utilisé pour la validation croisée et l'optimisation des hyperparamètres
#     (Optuna).
#
# internal_test_dynamic :
#   - conservé entièrement indépendant ;
#   - utilisé uniquement pour la validation interne finale des performances ;
#   - aucune information issue de ce jeu de données n'est utilisée pendant
#     l'entraînement ou le réglage du modèle.
# =============================================================================

DATA_DIR = "./data/My_data"
train_dynamic = pl.read_parquet(f"{DATA_DIR}/train_dynamic.parquet")
internal_test_dynamic = pl.read_parquet(f"{DATA_DIR}/internal_test_dynamic.parquet")


print("✅ Toutes les colonnes requises sont présentes.")
print(f"Jeu d'entrainement : {train_dynamic.shape}")
print(f"Jeu de test  : {internal_test_dynamic.shape}")
train_dynamic

#### Encodage des variables statiques

In [ ]:
# ==========================================================
# ENCODAGE DU SEXE
# Female -> 0
# Male   -> 1
# ==========================================================

train_dynamic = train_dynamic.with_columns(

    (
        pl.col("sex") == "Male"
    )
    .cast(pl.Float32)
    .alias("sex")

)

internal_test_dynamic = internal_test_dynamic.with_columns(

    (
        pl.col("sex") == "Male"
    )
    .cast(pl.Float32)
    .alias("sex")

)

# ==========================================================
# ENCODAGE ONE-HOT DU GÉNOTYPE
# ==========================================================

genetic_categories = [

    "other",
    "PKD2",
    "PKD1_non_truncating",
    "PKD1_truncating",
    "GUR"

]

train_dynamic = train_dynamic.with_columns([

    (
        pl.col("genetic") == category
    )
    .cast(pl.Float32)
    .alias(f"genetic_{category}")

    for category in genetic_categories

])

internal_test_dynamic = internal_test_dynamic.with_columns([

    (
        pl.col("genetic") == category
    )
    .cast(pl.Float32)
    .alias(f"genetic_{category}")

    for category in genetic_categories

])

# ==========================================================
# VARIABLES STATIQUES
# ==========================================================

static_columns = [

    # --------------------------------------------------
    # Sexe
    # --------------------------------------------------

    "sex",

    # --------------------------------------------------
    # Génotype
    # --------------------------------------------------

    "genetic_other",
    "genetic_PKD2",
    "genetic_PKD1_non_truncating",
    "genetic_PKD1_truncating",
    "genetic_GUR"

]

# ==========================================================
# CONSTRUCTION DU VECTEUR DE FEATURES STATIQUES
# ==========================================================

train_dynamic = train_dynamic.with_columns(

    pl.concat_list(
        static_columns
    )
    .alias("static_features")

)

internal_test_dynamic = internal_test_dynamic.with_columns(

    pl.concat_list(
        static_columns
    )
    .alias("static_features")

)


#### Normalisation

In [ ]:
# ==========================================================
# MOYENNES ET ÉCARTS-TYPES (TRAIN UNIQUEMENT)
# ==========================================================

all_age = np.concatenate(
    train_dynamic["age_history"].to_list()
)

all_egfr = np.concatenate(
    train_dynamic["egfr_history"].to_list()
)

all_years_ht = np.concatenate(
    train_dynamic["years_since_hypertension_history"].to_list()
)

all_years_uro = np.concatenate(
    train_dynamic["years_since_urologic_event_history"].to_list()
)

age_mean = all_age.mean()
age_std = all_age.std()

egfr_mean = all_egfr.mean()
egfr_std = all_egfr.std()

years_ht_mean = all_years_ht.mean()
years_ht_std = all_years_ht.std()

years_uro_mean = all_years_uro.mean()
years_uro_std = all_years_uro.std()

def normalize(values, mean, std):

    if std is None or std == 0:
        return values

    return [
        (float(x) - mean) / std
        for x in values
    ]

In [ ]:
# ==========================================================
# NORMALISATION DES HISTORIQUES (TRAIN)
# ==========================================================
# Les statistiques (moyenne et écart-type) sont calculées
# uniquement sur le jeu d'entraînement afin d'éviter toute
# fuite d'information (data leakage).
#
# Les variables binaires (0/1) ne sont PAS normalisées.
# Seules les variables continues sont transformées :
#   - age_history
#   - egfr_history
#   - years_since_hypertension_history
#   - years_since_urologic_event_history
# ==========================================================

train_dynamic = train_dynamic.with_columns(

    pl.col("age_history")

    .map_elements(

        lambda x: normalize(
            x,
            age_mean,
            age_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("age_history"),

    pl.col("egfr_history")

    .map_elements(

        lambda x: normalize(
            x,
            egfr_mean,
            egfr_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("egfr_history"),

    pl.col("years_since_hypertension_history")

    .map_elements(

        lambda x: normalize(
            x,
            years_ht_mean,
            years_ht_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("years_since_hypertension_history"),

    pl.col("years_since_urologic_event_history")

    .map_elements(

        lambda x: normalize(
            x,
            years_uro_mean,
            years_uro_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("years_since_urologic_event_history")

)


# ==========================================================
# NORMALISATION DES HISTORIQUES (TEST INTERNE)
# ==========================================================
# Les mêmes paramètres de normalisation calculés sur le jeu
# d'entraînement sont appliqués au jeu de test.
# ==========================================================

internal_test_dynamic = internal_test_dynamic.with_columns(

    pl.col("age_history")

    .map_elements(

        lambda x: normalize(
            x,
            age_mean,
            age_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("age_history"),

    pl.col("egfr_history")

    .map_elements(

        lambda x: normalize(
            x,
            egfr_mean,
            egfr_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("egfr_history"),

    pl.col("years_since_hypertension_history")

    .map_elements(

        lambda x: normalize(
            x,
            years_ht_mean,
            years_ht_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("years_since_hypertension_history"),

    pl.col("years_since_urologic_event_history")

    .map_elements(

        lambda x: normalize(
            x,
            years_uro_mean,
            years_uro_std
        ),

        return_dtype=pl.List(pl.Float64)

    )

    .alias("years_since_urologic_event_history")

)

In [ ]:
# ==========================================================
# CONSTRUCTION DES SNAPSHOTS POUR LE CDE
# ==========================================================
# Chaque snapshot correspond à un patient observé jusqu'à une
# visite donnée.
#
# Les variables longitudinales sont converties en tableaux
# NumPy afin d'être utilisées par le modèle Neural CDE.
#
# Les variables statiques (sexe, génétique, etc.) sont
# regroupées dans "static".
# ==========================================================

def build_snapshots_from_polars(df):

    samples = []

    for row in df.iter_rows(named=True):

        samples.append({

            "patient_id":
                row["patient_id"],

            # ==========================================
            # TEMPS
            # ==========================================

            "time_history":
                np.asarray(
                    row["time_history"],
                    dtype=np.float32
                ),

            # ==========================================
            # VARIABLES LONGITUDINALES CONTINUES
            # ==========================================

            "age_history":
                np.asarray(
                    row["age_history"],
                    dtype=np.float32
                ),

            "egfr_history":
                np.asarray(
                    row["egfr_history"],
                    dtype=np.float32
                ),

            "years_since_hypertension_history":
                np.asarray(
                    row["years_since_hypertension_history"],
                    dtype=np.float32
                ),

            "years_since_urologic_event_history":
                np.asarray(
                    row["years_since_urologic_event_history"],
                    dtype=np.float32
                ),

            # ==========================================
            # VARIABLES LONGITUDINALES BINAIRES
            # ==========================================

            "hypertension_history":
                np.asarray(
                    row["hypertension_history"],
                    dtype=np.float32
                ),

            "hypertension_date_known_history":
                np.asarray(
                    row["hypertension_date_known_history"],
                    dtype=np.float32
                ),

            "urologic_history":
                np.asarray(
                    row["urologic_history"],
                    dtype=np.float32
                ),

            "urologic_date_known_history":
                np.asarray(
                    row["urologic_date_known_history"],
                    dtype=np.float32
                ),

            # ==========================================
            # VARIABLES STATIQUES
            # ==========================================

            "static":
                np.asarray(
                    row["static_features"],
                    dtype=np.float32
                ),

            # ==========================================
            # CIBLE
            # ==========================================

            "time_to_event":
                np.float32(
                    row["time_to_event"]
                ),

            "event":
                np.float32(
                    row["event_irt"]
                )

        })

    return samples

In [ ]:
# ==========================================================
# CONSTRUCTION DES SNAPSHOTS
# ==========================================================

train_samples = build_snapshots_from_polars(
    train_dynamic
)

test_samples = build_snapshots_from_polars(
    internal_test_dynamic
)

print(f"Train snapshots : {len(train_samples)}")
print(f"Test snapshots  : {len(test_samples)}")

## Étape 2 — Construction de la trajectoire contrôlée

Chaque snapshot correspond à l'historique clinique d'un patient jusqu'à une visite donnée. Les trajectoires peuvent contenir un nombre de visites différent : elles sont donc **complétées (padding)** jusqu'à une longueur commune `max_len` afin de permettre le traitement vectorisé (`vmap`) des batches.

À chaque instant d'observation, le signal de contrôle regroupe les variables longitudinales disponibles (temps, âge, eGFR, hypertension, complications urologiques et leurs caractéristiques), tandis que les variables statiques sont traitées séparément.

Le padding est réalisé uniquement pour homogénéiser les dimensions des tenseurs. Lors de l'intégration du Neural CDE, Diffrax utilise le **temps réel de la dernière visite** (`t_end`) de chaque patient et ignore ainsi les points ajoutés artificiellement.

In [ ]:
# ==========================================================
# PRÉPARATION DES TENSEURS POUR LE NEURAL CDE
# ==========================================================
#
# Cette fonction transforme les snapshots en tenseurs
# compatibles avec Diffrax.
#
# Étapes :
#   1. Détermination de la longueur maximale des trajectoires.
#   2. Padding des trajectoires pour obtenir une longueur commune.
#   3. Construction des canaux longitudinaux.
#   4. Calcul des coefficients d'interpolation d'Hermite.
#   5. Préparation des variables statiques et des cibles.
#
# Les canaux longitudinaux sont :
#
#   0 : temps
#   1 : âge
#   2 : eGFR
#   3 : hypertension
#   4 : date HTA connue
#   5 : années depuis l'hypertension
#   6 : complication urologique
#   7 : date complication connue
#   8 : années depuis la complication
# ==========================================================

def prepare_tensors(samples):

    # --------------------------------------------------
    # Informations générales
    # --------------------------------------------------

    max_len = max(
        len(s["time_history"])
        for s in samples
    )

    static_dim = samples[0]["static"].shape[0]

    n = len(samples)

    # --------------------------------------------------
    # Variables longitudinales utilisées par le CDE
    # --------------------------------------------------

    longitudinal_keys = [

        "time_history",

        "age_history",

        "egfr_history",

        "hypertension_history",

        "hypertension_date_known_history",

        "years_since_hypertension_history",

        "urologic_history",

        "urologic_date_known_history",

        "years_since_urologic_event_history",

    ]

    n_channels = len(longitudinal_keys)

    # --------------------------------------------------
    # Allocation mémoire
    # --------------------------------------------------

    ts_padded = np.full(
        (n, max_len),
        np.nan,
        dtype=np.float32
    )

    ys_padded = np.full(
        (n, max_len, n_channels),
        np.nan,
        dtype=np.float32
    )

    t_end = np.zeros(
        (n,),
        dtype=np.float32
    )

    static_arr = np.zeros(
        (n, static_dim),
        dtype=np.float32
    )

    tte_arr = np.zeros(
        (n,),
        dtype=np.float32
    )

    event_arr = np.zeros(
        (n,),
        dtype=np.float32
    )

    n_visits_arr = np.zeros(
        (n,),
        dtype=np.int32
    )

    patient_ids = []

    # --------------------------------------------------
    # Construction des trajectoires
    # --------------------------------------------------

    for i, s in enumerate(samples):

        k = len(s["time_history"])

        t_real = s["time_history"].astype(np.float32)

        # ----------------------------------------------
        # Padding temporel
        # ----------------------------------------------

        if k < max_len:

            pad = (
                t_real[-1]
                + np.arange(
                    1,
                    max_len - k + 1,
                    dtype=np.float32
                ) * 1e-3
            )

            ts_full = np.concatenate([
                t_real,
                pad
            ])

        else:

            ts_full = t_real

        ts_padded[i] = ts_full

        # ----------------------------------------------
        # Construction des canaux longitudinaux
        # ----------------------------------------------

        for channel, key in enumerate(longitudinal_keys):

            ys_padded[i, :k, channel] = np.asarray(
                s[key],
                dtype=np.float32
            )

        # ----------------------------------------------
        # Cas particulier :
        # une seule visite disponible
        #
        # Duplication nécessaire pour permettre
        # l'interpolation d'Hermite.
        # ----------------------------------------------

        if k == 1 and max_len > 1:

            ys_padded[i, 1, :] = ys_padded[i, 0, :]

        # ----------------------------------------------
        # Variables associées au snapshot
        # ----------------------------------------------

        t_end[i] = t_real[-1]

        static_arr[i] = s["static"]

        tte_arr[i] = s["time_to_event"]

        event_arr[i] = s["event"]

        n_visits_arr[i] = k

        patient_ids.append(
            s["patient_id"]
        )

    # --------------------------------------------------
    # Calcul des coefficients d'interpolation
    # d'Hermite utilisés par Diffrax
    # --------------------------------------------------

    ts_padded = jnp.array(ts_padded)

    ys_padded = jnp.array(ys_padded)

    coeffs = jax.vmap(

        lambda ts, ys:
            diffrax.backward_hermite_coefficients(
                ts,
                ys,
                fill_forward_nans_at_end=True
            )

    )(ts_padded, ys_padded)

    # --------------------------------------------------
    # Retour
    # --------------------------------------------------

    return {

        "ts":
            ts_padded,

        "coeffs":
            coeffs,

        "t_end":
            jnp.array(t_end),

        "static":
            jnp.array(static_arr),

        "tte":
            jnp.array(tte_arr),

        "event":
            jnp.array(event_arr),

        "n_visits":
            n_visits_arr,

        "patient_ids":
            np.array(patient_ids),

        "static_dim":
            static_dim,

        "max_len":
            max_len,
        "n_channels": 
            n_channels,

    }

In [ ]:
# ==========================================================
# PRÉPARATION DES TENSEURS DU JEU D'ENTRAÎNEMENT
# ==========================================================

train_data = prepare_tensors(
    train_samples
)

# ==========================================================
# PRÉPARATION DES TENSEURS DU JEU DE TEST INTERNE
# ==========================================================

test_data = prepare_tensors(
    test_samples
)

# ==========================================================
# VÉRIFICATION DES DIMENSIONS
# ==========================================================
# Affiche le nombre de snapshots, la longueur maximale des
# trajectoires après padding et la dimension des variables
# statiques.

print(
    "Train :", len(train_samples),
    "| max_len :", train_data["max_len"],
    "| static_dim :", train_data["static_dim"]
)

print(
    "Test  :", len(test_samples),
    "| max_len :", test_data["max_len"],
    "| static_dim :", test_data["static_dim"]
)

print(
    "Train coeffs shape :",
    jax.tree_util.tree_leaves(train_data["coeffs"])[0].shape
)

print(
    "Test coeffs shape :",
    jax.tree_util.tree_leaves(test_data["coeffs"])[0].shape
)

#### Découpage train / validation  **par patient**

In [ ]:
# ==========================================================
# SPLIT DES PATIENTS (TRAIN / VALIDATION)
# ==========================================================
# La séparation est effectuée au niveau des patients et non
# des snapshots afin d'éviter toute fuite d'information.
#
# Ainsi, tous les snapshots d'un même patient appartiennent
# exclusivement au jeu d'entraînement ou au jeu de validation.
# ==========================================================

def split_by_patient(
    patient_ids,
    train_frac=0.8,
    seed=42
):

    rng = np.random.default_rng(seed)

    # --------------------------------------------------
    # Liste des patients uniques
    # --------------------------------------------------

    unique_patients = np.unique(patient_ids)

    rng.shuffle(unique_patients)

    # --------------------------------------------------
    # Nombre de patients conservés
    # pour l'entraînement
    # --------------------------------------------------

    n_train = int(
        train_frac * len(unique_patients)
    )

    # --------------------------------------------------
    # Répartition des patients
    # --------------------------------------------------

    train_patients = set(
        unique_patients[:n_train]
    )

    val_patients = set(
        unique_patients[n_train:]
    )

    # --------------------------------------------------
    # Indices des snapshots appartenant
    # aux patients d'entraînement
    # --------------------------------------------------

    train_idx = np.array([

        i

        for i, pid in enumerate(patient_ids)

        if pid in train_patients

    ])

    # --------------------------------------------------
    # Indices des snapshots appartenant
    # aux patients de validation
    # --------------------------------------------------

    val_idx = np.array([

        i

        for i, pid in enumerate(patient_ids)

        if pid in val_patients

    ])

    return train_idx, val_idx


# ==========================================================
# CONSTRUCTION DES JEUX D'ENTRAÎNEMENT ET DE VALIDATION
# ==========================================================

train_idx, val_idx = split_by_patient(

    train_data["patient_ids"],

    train_frac=0.8,

    seed=42

)

# ==========================================================
# VÉRIFICATION DE LA RÉPARTITION
# ==========================================================

print(
    f"Train snapshots      : {len(train_idx)}"
)

print(
    f"Validation snapshots : {len(val_idx)}"
)

train_patients = set(train_data["patient_ids"][train_idx])
val_patients = set(train_data["patient_ids"][val_idx])

print(
    "Patients communs :",
    len(train_patients & val_patients)
)

## Étapes 3, 4 et 5 — Modèle Neural CDE et tête de survie

Cette étape définit l'architecture complète du modèle de survie basée sur un **Neural Controlled Differential Equation (Neural CDE)**. Le modèle apprend une représentation continue de l'évolution clinique du patient à partir de sa trajectoire longitudinale, puis combine cette représentation avec les variables statiques afin de prédire le temps avant l'initiation du traitement de suppléance rénale.

L'architecture est composée de trois blocs principaux :

- **Champ de vecteurs (`CDEVectorField`)** : un perceptron multicouche (MLP) apprend le champ de vecteurs \(f_\theta\) intervenant dans l'équation
  \[
  dz(t)=f_\theta(z(t))\,dX(t).
  \]
  Le réseau utilise une activation **Softplus** dans les couches cachées et une activation **Tanh** en sortie afin de limiter l'amplitude du champ de vecteurs. Un facteur d'échelle (`scale = 0.1`) est ensuite appliqué pour stabiliser l'intégration numérique et éviter des dynamiques trop brusques.

- **Neural CDE (`NeuralCDESurvival`)** : la trajectoire clinique interpolée est construite à l'aide d'une interpolation cubique (`CubicInterpolation`). L'état latent initial est obtenu en projetant la première observation dans l'espace latent, puis il est propagé le long de la trajectoire grâce au solveur adaptatif **Tsit5** (`ControlTerm` + `PIDController`). L'intégration est réalisée uniquement jusqu'au **temps réel de la dernière visite** (`t_end`) afin de ne jamais utiliser les points ajoutés lors du padding.

- **Tête de survie** : l'état latent final est concaténé aux variables statiques du patient, puis transmis à un second MLP (`Linear → ReLU → Linear`) qui prédit les paramètres \((\mu,\log\sigma)\) de la distribution log-normale utilisée pour modéliser le temps avant l'événement.

Enfin, un cas particulier est traité lorsque le patient ne possède qu'une seule visite (\(t_0=t_1\)). Dans cette situation, l'intégration d'une équation différentielle sur un intervalle nul peut déstabiliser le contrôleur adaptatif (`PIDController`). Le modèle utilise donc `jax.lax.cond` pour contourner cette situation et retourne directement l'état latent initial \(z_0\), sans effectuer d'intégration.

In [ ]:
# ==========================================================
# CHAMP DE VECTEURS DU NEURAL CDE
# ==========================================================
#
# Cette classe représente le champ de vecteurs fθ utilisé
# dans l'équation différentielle contrôlée :
#
#           dz(t) = fθ(z(t)) dX(t)
#
# où :
#   • X(t) est la trajectoire clinique interpolée,
#   • z(t) est l'état latent,
#   • fθ est appris par un réseau de neurones.
#
# Le MLP produit une matrice de taille
#
#       hidden_size × data_size
#
# qui décrit comment chaque dimension de l'état latent est
# influencée par chacun des canaux de la trajectoire.
# ==========================================================

class CDEVectorField(eqx.Module):

    mlp: eqx.nn.MLP

    data_size: int

    hidden_size: int

    scale: float

    def __init__(
        self,
        data_size,
        hidden_size,
        width_size,
        depth,
        scale=0.1,
        *,
        key,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.data_size = data_size

        self.hidden_size = hidden_size

        self.scale = scale

        self.mlp = eqx.nn.MLP(

            in_size=hidden_size,

            out_size=hidden_size * data_size,

            width_size=width_size,

            depth=depth,

            activation=jnn.softplus,

            # tanh borne la sortie et stabilise
            # l'intégration numérique.
            final_activation=jnn.tanh,

            key=key,

        )

    def __call__(self, t, y, args):

        field = self.mlp(y).reshape(

            self.hidden_size,

            self.data_size

        )

        return self.scale * field


# ==========================================================
# MODÈLE NEURAL CDE POUR L'ANALYSE DE SURVIE
# ==========================================================
#
# Ce modèle apprend l'évolution continue de l'état latent
# d'un patient à partir de sa trajectoire clinique.
#
# Architecture :
#
#      Trajectoire contrôlée X(t),
#      MLP d'initialisation,
#          État latent initial z0,
#        Intégration du Neural CDE,
#         État latent final z(T)( + Variables statiques),
#          Tête de survie (MLP),
#          Paramètres (μ, σ)
#
# Le modèle prédit les paramètres d'une distribution
# log-normale utilisée pour estimer le temps avant
# l'initiation du traitement de suppléance rénale.
# ==========================================================

class NeuralCDESurvival(eqx.Module):

    initial: eqx.nn.MLP

    func: CDEVectorField

    surv_head: eqx.nn.MLP

    hidden_size: int

    static_dim: int

    def __init__(
        self,
        data_size,
        hidden_size,
        static_dim,
        width_size,
        depth,
        surv_width,
        surv_depth=2,
        *,
        key,
        **kwargs
    ):

        super().__init__(**kwargs)

        ikey, fkey, skey = jr.split(key, 3)

        self.hidden_size = hidden_size

        self.static_dim = static_dim

        # --------------------------------------------------
        # Projection de la première observation
        # vers l'espace latent.
        # --------------------------------------------------

        self.initial = eqx.nn.MLP(

            in_size=data_size,

            out_size=hidden_size,

            width_size=width_size,

            depth=depth,

            key=ikey,

        )

        # --------------------------------------------------
        # Champ de vecteurs du Neural CDE.
        # --------------------------------------------------

        self.func = CDEVectorField(

            data_size=data_size,

            hidden_size=hidden_size,

            width_size=width_size,

            depth=depth,

            key=fkey,

        )

        # --------------------------------------------------
        # Tête de survie.
        #
        # Entrée :
        #   état latent final + variables statiques
        #
        # Sortie :
        #   μ et log(σ)
        # --------------------------------------------------

        self.surv_head = eqx.nn.MLP(

            in_size=hidden_size + static_dim,

            out_size=2,

            width_size=surv_width,

            depth=surv_depth,

            activation=jnn.relu,

            key=skey,

        )

    def __call__(

        self,

        ts,

        coeffs,

        t1,

        static

    ):

        # ==================================================
        # ÉTAPE 1 — Construction de la trajectoire contrôlée
        # ==================================================

        control = diffrax.CubicInterpolation(

            ts,

            coeffs

        )

        # ==================================================
        # ÉTAPE 2 — Construction du système différentiel
        #
        #          dz = f(z)dX
        # ==================================================

        term = diffrax.ControlTerm(

            self.func,

            control

        ).to_ode()

        solver = diffrax.Tsit5()

        # ==================================================
        # ÉTAPE 3 — Initialisation de l'état latent
        #
        # Le premier point de la trajectoire est projeté
        # dans l'espace latent.
        # ==================================================

        y0 = self.initial(

            control.evaluate(ts[0])

        )

        # ==================================================
        # ÉTAPE 4 — Intégration du Neural CDE
        #
        # L'intégration s'arrête au temps réel de la
        # dernière visite (t1), sans utiliser les points
        # ajoutés lors du padding.
        # ==================================================

        def integrate(_):

            solution = diffrax.diffeqsolve(

                term,

                solver,

                ts[0],

                t1,

                None,

                y0,

                stepsize_controller=diffrax.PIDController(

                    rtol=1e-3,

                    atol=1e-6,

                ),

                saveat=diffrax.SaveAt(

                    t1=True

                ),

                max_steps=4096,

            )

            return solution.ys[-1]

        # ==================================================
        # Cas particulier :
        # une seule visite disponible.
        #
        # Aucune intégration n'est réalisée ; l'état latent
        # est simplement l'état initial.
        # ==================================================

        def skip(_):

            return y0

        single_visit = (

            t1 <= ts[0] + 1e-4

        )

        latent_state = jax.lax.cond(

            single_visit,

            skip,

            integrate,

            operand=None,

        )

        # ==================================================
        # ÉTAPE 5 — Fusion avec les variables statiques
        # ==================================================

        combined = jnp.concatenate(

            [

                latent_state,

                static,

            ],

            axis=-1,

        )

        # ==================================================
        # ÉTAPE 6 — Prédiction des paramètres de survie
        #
        # μ      : moyenne de log(T)
        # σ > 0  : écart-type de log(T)
        # ==================================================

        out = self.surv_head(

            combined

        )

        mu = out[0]

        log_sigma = out[1]

        sigma = jnn.softplus(

            log_sigma

        ) + 1e-6

        # Évite les valeurs numériquement instables.

        sigma = jnp.clip(

            sigma,

            1e-3,

            20.0,

        )

        return mu, sigma

## Étape 6 — Perte Log-Normale AFT


In [ ]:
# ==========================================================
# FONCTION DE PERTE LOG-NORMALE AFT
# ==========================================================
#
# Le modèle prédit les paramètres (μ, σ) d'une distribution
# log-normale représentant le temps avant l'événement.
#
# Pour chaque patient :
#
#   • si l'événement est observé (event = 1),
#     la vraisemblance est la densité :
#
#         log f(t)
#
#   • si l'observation est censurée (event = 0),
#     seule la probabilité de survie est connue :
#
#         log S(t)
#
# La fonction retourne la moyenne de la log-vraisemblance
# négative (Negative Log-Likelihood, NLL), utilisée comme
# fonction objectif pendant l'entraînement.
# ==========================================================

def aft_loss(

    mu,

    sigma,

    time_to_event,

    event,

):

    # --------------------------------------------------
    # Protection numérique
    # --------------------------------------------------

    eps = 1e-6

    t = jnp.clip(

        time_to_event,

        eps,

        None,

    )

    sigma = jnp.clip(

        sigma,

        eps,

        None,

    )

    # --------------------------------------------------
    # Variable réduite de la loi log-normale
    #
    #          log(t) - μ
    #   z = ----------------
    #               σ
    # --------------------------------------------------

    z = (

        jnp.log(t) - mu

    ) / sigma

    # --------------------------------------------------
    # Log-densité :
    #
    # log f(t)
    # --------------------------------------------------

    log_pdf = (

        -jnp.log(t)

        - jnp.log(sigma)

        - 0.5 * jnp.log(

            2.0 * jnp.pi

        )

        - 0.5 * z ** 2

    )

    # --------------------------------------------------
    # Log-fonction de survie :
    #
    # log S(t)
    #
    # Utilisation de log_ndtr pour une meilleure stabilité
    # numérique lorsque z est grand.
    # --------------------------------------------------

    log_surv = jax.scipy.special.log_ndtr(

        -z

    )

    # --------------------------------------------------
    # Contribution de chaque observation
    #
    # Événement observé :
    #     log f(t)
    #
    # Observation censurée :
    #     log S(t)
    # --------------------------------------------------

    log_likelihood = jnp.where(

        event == 1,

        log_pdf,

        log_surv,

    )

    # --------------------------------------------------
    # Negative Log-Likelihood moyenne
    # --------------------------------------------------

    loss = -jnp.mean(

        log_likelihood

    )

    return loss

## Entraînement


In [ ]:
# ==========================================================
# GÉNÉRATEUR DE MINI-BATCHES
# ==========================================================
#
# Génère des mini-batches de snapshots d'entraînement.
#
# À chaque époque :
#   • les indices sont mélangés aléatoirement ;
#   • chaque snapshot apparaît une seule fois ;
#   • seuls les indices d'entraînement sont utilisés.
#
# Les snapshots de validation ne sont jamais vus pendant
# l'optimisation, ce qui garantit l'absence de fuite
# d'information.
# ==========================================================

def dataloader(

    idx_array,

    batch_size,

    *,

    key,

):

    n = len(idx_array)

    while True:

        key, subkey = jr.split(key)

        permutation = jr.permutation(

            subkey,

            jnp.array(idx_array)

        )

        start = 0

        while start + batch_size <= n:

            yield permutation[

                start : start + batch_size

            ]

            start += batch_size


# ==========================================================
# FONCTION DE PERTE DIFFÉRENTIABLE
# ==========================================================
#
# Pour chaque mini-batch :
#
#      Trajectoires
#           │
#           ▼
#      Neural CDE
#           │
#           ▼
#        (μ, σ)
#           │
#           ▼
#   Log-normal AFT loss
#
# Les gradients sont calculés automatiquement grâce à JAX.
# ==========================================================

def make_loss_fn():

    @eqx.filter_value_and_grad
    def loss_fn(

        model,

        ts,

        coeffs,

        t1,

        static,

        tte,

        event,

    ):

        mu, sigma = jax.vmap(model)(

            ts,

            coeffs,

            t1,

            static,

        )

        return aft_loss(

            mu,

            sigma,

            tte,

            event,

        )

    return eqx.filter_jit(loss_fn)


# ==========================================================
# ENTRAÎNEMENT DU MODÈLE
# ==========================================================
#
# Le modèle est entraîné uniquement sur les snapshots
# d'entraînement.
#
# Les snapshots de validation sont utilisés exclusivement
# pour :
#
#   • suivre la performance,
#   • sélectionner le meilleur modèle,
#   • déclencher l'early stopping.
#
# Aucun gradient n'est calculé sur la validation.
# ==========================================================

def train_model(

    data,

    train_idx,

    val_idx,

    hidden_size=64,

    width_size=64,

    depth=2,

    surv_width=64,

    surv_depth=2,

    lr=1e-3,

    n_steps=200,

    batch_size=64,

    patience=20,

    seed=42,

    verbose=True,

):

    # --------------------------------------------------
    # Initialisation
    # --------------------------------------------------

    key = jr.PRNGKey(seed)

    model_key, loader_key = jr.split(key)

    model = NeuralCDESurvival(

        data_size=data["n_channels"],

        hidden_size=hidden_size,

        static_dim=data["static_dim"],

        width_size=width_size,

        depth=depth,

        surv_width=surv_width,

        surv_depth=surv_depth,

        key=model_key,

    )

    grad_loss = make_loss_fn()

    optimizer = optax.chain(

        optax.clip_by_global_norm(1.0),

        optax.adamw(

            learning_rate=lr,

            weight_decay=1e-4,

        ),

    )

    opt_state = optimizer.init(

        eqx.filter(

            model,

            eqx.is_inexact_array,

        )

    )

    # --------------------------------------------------
    # Évaluation sur un ensemble de snapshots
    #
    # Aucun gradient n'est calculé.
    # Cette fonction est utilisée uniquement pour
    # l'early stopping.
    # --------------------------------------------------

    def evaluate_loss(

        model,

        idx,

    ):

        coeffs = tuple(

            x[idx]

            for x in data["coeffs"]

        )

        mu, sigma = jax.vmap(model)(

            data["ts"][idx],

            coeffs,

            data["t_end"][idx],

            data["static"][idx],

        )

        loss = aft_loss(

            mu,

            sigma,

            data["tte"][idx],

            data["event"][idx],

        )

        return float(loss)

    # --------------------------------------------------
    # Initialisation de l'early stopping
    # --------------------------------------------------

    best_val_loss = float("inf")

    best_model = model

    patience_counter = 0

    loader = dataloader(

        train_idx,

        batch_size,

        key=loader_key,

    )

    eval_every = 10

    # --------------------------------------------------
    # Boucle d'entraînement
    # --------------------------------------------------

    for step in range(n_steps):

        batch_idx = next(loader)

        coeffs = tuple(

            x[batch_idx]

            for x in data["coeffs"]

        )

        train_loss, grads = grad_loss(

            model,

            data["ts"][batch_idx],

            coeffs,

            data["t_end"][batch_idx],

            data["static"][batch_idx],

            data["tte"][batch_idx],

            data["event"][batch_idx],

        )

        # ----------------------------------------------
        # Mise à jour des paramètres (AdamW)
        # ----------------------------------------------

        updates, opt_state = optimizer.update(

            grads,

            opt_state,

            params=eqx.filter(

                model,

                eqx.is_inexact_array,

            ),

        )

        model = eqx.apply_updates(

            model,

            updates,

        )

        # ----------------------------------------------
        # Évaluation sur le jeu de validation
        # ----------------------------------------------

        if step % eval_every == 0 or step == n_steps - 1:

            val_loss = evaluate_loss(

                model,

                val_idx,

            )

            if verbose:

                print(

                    f"Step {step:03d} | "

                    f"Train = {float(train_loss):.4f} | "

                    f"Val = {val_loss:.4f}"

                )

            # ------------------------------------------
            # Sauvegarde du meilleur modèle
            # ------------------------------------------

            if val_loss < best_val_loss:

                best_val_loss = val_loss

                best_model = model

                patience_counter = 0

            else:

                patience_counter += 1

                # --------------------------------------
                # Early stopping
                # --------------------------------------

                if patience_counter >= patience:

                    if verbose:

                        print("\nEarly stopping")

                    break

    return best_model, best_val_loss

## Étape 7 — Optimisation des hyperparamètres (Optuna)



In [ ]:
# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES AVEC OPTUNA
# ==========================================================
#
# Les hyperparamètres du Neural CDE sont optimisés en
# utilisant uniquement le jeu d'entraînement.
#
# Pour chaque essai :
#
#   • un nouvel ensemble d'hyperparamètres est proposé ;
#   • le modèle est entraîné sur les snapshots
#     d'entraînement ;
#   • la performance est évaluée sur les snapshots de
#     validation ;
#   • la validation loss est renvoyée à Optuna.
#
# Le jeu de test n'est jamais utilisé durant cette étape.
# ==========================================================

N_TRIALS = 50


def objective_fn(trial):

    hidden_size = trial.suggest_categorical(
        "hidden_size",
        [32, 64, 128],
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True,
    )

    width_size = trial.suggest_categorical(
        "width_size",
        [32, 64, 128],
    )

    depth = trial.suggest_int(
        "depth",
        1,
        3,
    )

    surv_width = trial.suggest_categorical(
        "surv_width",
        [32, 64, 128],
    )

    _, val_loss = train_model(

        train_data,

        train_idx,

        val_idx,

        hidden_size=hidden_size,

        width_size=width_size,

        depth=depth,

        surv_width=surv_width,

        lr=learning_rate,

        n_steps=200,

        batch_size=64,

        patience=20,

        seed=42,

        verbose=False,

    )

    return val_loss


sampler = optuna.samplers.TPESampler(
    seed=42,
)

study = optuna.create_study(

    direction="minimize",

    sampler=sampler,

)

t0 = time.time()

study.optimize(

    objective_fn,

    n_trials=N_TRIALS,

)

elapsed = time.time() - t0

print(
    f"\nOptimisation terminée en {elapsed:.1f} secondes."
)

print(
    f"Temps moyen par essai : {elapsed / N_TRIALS:.1f} secondes."
)

print("\nMeilleurs hyperparamètres :")

print(study.best_params)

print("\nMeilleure validation loss :")

print(study.best_value)

### Entraînement final avec les meilleurs hyperparamètres

In [ ]:
# ==========================================================
# ENTRAÎNEMENT DU MODÈLE FINAL
# ==========================================================
#
# Le modèle est réentraîné à partir des meilleurs
# hyperparamètres trouvés par Optuna.
#
# Le jeu de test n'est toujours pas utilisé à cette étape.
# Il est conservé pour l'évaluation finale.
# ==========================================================

best_params = study.best_params

print("\nHyperparamètres retenus :")

for key, value in best_params.items():

    print(f"{key:<15}: {value}")

final_model, final_val_loss = train_model(

    train_data,

    train_idx,

    val_idx,

    hidden_size=best_params["hidden_size"],

    width_size=best_params["width_size"],

    depth=best_params["depth"],

    surv_width=best_params["surv_width"],

    lr=best_params["learning_rate"],

    n_steps=500,

    batch_size=64,

    patience=20,

    seed=42,

    verbose=True,

)

print(

    "\nValidation loss finale :",

    final_val_loss,

)

## Étape 8 — Évaluation : C-index, MAE, RMSE, par nombre de visites



In [ ]:
# ==========================================================
# PRÉDICTION
# ==========================================================
#
# Applique le modèle entraîné à un ensemble de snapshots.
#
# Le modèle prédit les paramètres (μ, σ) de la distribution
# log-normale puis en déduit le temps médian avant
# l'événement :
#
#          médiane(T) = exp(μ)
#
# Aucun gradient n'est calculé durant cette étape.
# ==========================================================

def predict(

    model,

    data,

):

    # --------------------------------------------------
    # Sélection de tous les snapshots
    # --------------------------------------------------

    n_samples = data["ts"].shape[0]

    sample_idx = np.arange(

        n_samples

    )

    # --------------------------------------------------
    # Coefficients d'interpolation
    # --------------------------------------------------

    coeffs = tuple(

        x[sample_idx]

        for x in data["coeffs"]

    )

    # --------------------------------------------------
    # Prédiction des paramètres de survie
    # --------------------------------------------------

    mu, sigma = jax.vmap(model)(

        data["ts"],

        coeffs,

        data["t_end"],

        data["static"],

    )

    mu = np.asarray(mu)

    sigma = np.asarray(sigma)

    # --------------------------------------------------
    # Conversion de la distribution prédite en une
    # estimation ponctuelle du temps restant avant l'IRT.
    #
    # Le modèle prédit une distribution log-normale
    # caractérisée par (μ, σ). Pour obtenir une valeur
    # unique, on utilise ici la médiane :
    #
    #       predicted_time = exp(μ)
    # --------------------------------------------------

    predicted_time = np.exp(mu)

    return (

        predicted_time,

        mu,

        sigma,

    )

In [ ]:
# ==========================================================
# ÉVALUATION DU MODÈLE
# ==========================================================
#
# Cette fonction évalue le modèle entraîné sur un jeu de
# données (validation ou test).
#
# Les métriques calculées sont :
#
#   • C-index
#   • MAE
#   • RMSE
#
# Les erreurs sont calculées uniquement chez les patients
# ayant atteint l'IRT (event = 1), puisque le temps réel
# jusqu'à l'événement est inconnu pour les patients censurés.
# ==========================================================

def evaluate(model, data, label="Test"):

    predicted_time, mu, sigma = predict(

        model,

        data

    )

    observed_time = np.array(

        data["tte"]

    )

    event = np.array(

        data["event"]

    )

    n_visits = np.array(

        data["n_visits"]

    )

    # --------------------------------------------------
    # Patients ayant effectivement atteint l'IRT
    #
    # Les métriques de régression (MAE, RMSE) ne peuvent
    # être calculées que lorsque le temps réel jusqu'à
    # l'événement est connu.
    # --------------------------------------------------

    mask = event == 1

    # --------------------------------------------------
    # Indice de concordance
    #
    # Le C-index mesure la capacité du modèle à ordonner
    # correctement les patients selon leur risque.
    # --------------------------------------------------

    c_index = concordance_index(

        observed_time,

        mu,

        event

    )

    print(f"\n===== {label} =====")

    print(f"C-index : {c_index:.4f}")

    # --------------------------------------------------
    # Erreurs de prédiction par nombre de visites
    # --------------------------------------------------

    results = []

    print("\nPar nombre de visites :")

    for n in sorted(np.unique(n_visits[mask])):

        idx = mask & (n_visits == n)

        if idx.sum() < 5:

            continue

        mae_n = mean_absolute_error(

            observed_time[idx],

            predicted_time[idx]

        )

        rmse_n = np.sqrt(

            mean_squared_error(

                observed_time[idx],

                predicted_time[idx]

            )

        )

        print(

            f"{n} visites | "

            f"n={idx.sum()} | "

            f"MAE={mae_n:.3f} | "

            f"RMSE={rmse_n:.3f}"

        )

        results.append({

            "n_visits": int(n),

            "n_events": int(idx.sum()),

            "mae": mae_n,

            "rmse": rmse_n

        })

    return {

        "predicted_time": predicted_time,

        "observed_time": observed_time,

        "event": event,

        "n_visits": n_visits,

        "mu": mu,

        "sigma": sigma,

        "c_index": c_index,

        "per_visit": results

    }

In [ ]:
results_test = evaluate(

    final_model,

    test_data,

    label="Test"

)

In [ ]:
# ==========================================================
# CONSTRUCTION DU DATAFRAME DES PRÉDICTIONS
# ==========================================================
#
# Construit un DataFrame regroupant les prédictions du modèle
# ainsi que les informations observées pour chaque snapshot.
#
# Ce tableau facilite :
#   • l'analyse des performances du modèle ;
#   • la création de graphiques ;
#   • les analyses par sous-groupes (nombre de visites,
#     censure, etc.).
#
# Le modèle prédit une distribution log-normale du temps
# restant avant l'IRT, caractérisée par les paramètres
# (μ, σ). L'estimation ponctuelle du temps restant utilisée
# ici correspond à la médiane de cette distribution :
#
#           predicted_time = exp(μ)
# ==========================================================

def build_prediction_dataframe(model, data):

    # --------------------------------------------------
    # Prédictions du modèle
    # --------------------------------------------------

    predicted_time, mu, sigma = predict(

        model,

        data,

    )

    # --------------------------------------------------
    # Construction du tableau récapitulatif
    # --------------------------------------------------

    predictions = pd.DataFrame({

        "patient_id": data["patient_ids"],

        "n_measures": data["n_visits"],

        "time_to_event": data["tte"],

        "predicted_time": predicted_time,

        "mu": mu,

        "sigma": sigma,

        "event": data["event"],

    })

    # --------------------------------------------------
    # Erreurs de prédiction
    # --------------------------------------------------

    predictions["error"] = (

        predictions["predicted_time"]

        - predictions["time_to_event"]

    )

    predictions["absolute_error"] = np.abs(

        predictions["error"]

    )

    return predictions

In [ ]:
pred_cde = build_prediction_dataframe(

    final_model,

    test_data

)

pred_cde.head(15)

In [ ]:
# ==========================================================
# PRÉDICTIONS D'UN PATIENT
# ==========================================================
#
# Affiche l'évolution des prédictions du modèle pour un
# patient donné au fil de son suivi.
#
# Chaque ligne correspond à un snapshot construit à partir
# d'une visite clinique et contient :
#
#   • le nombre de visites disponibles ;
#   • le temps restant observé avant l'IRT ;
#   • le temps restant prédit par le modèle ;
#   • le statut de l'événement.
#
# Les résultats sont triés par nombre de visites afin de
# visualiser l'évolution des prédictions lorsque davantage
# d'informations longitudinales deviennent disponibles.
# ==========================================================

def show_patient_predictions(

    patient_id,

    results,

    data,

):

    # --------------------------------------------------
    # Recherche des snapshots du patient
    # --------------------------------------------------

    patient_ids = np.asarray(

        data["patient_ids"]

    )

    patient_idx = np.where(

        patient_ids == patient_id

    )[0]

    if len(patient_idx) == 0:

        print(

            f"Patient '{patient_id}' introuvable."

        )

        return None

    # --------------------------------------------------
    # Construction du tableau récapitulatif
    # --------------------------------------------------

    patient_predictions = pd.DataFrame({

        "n_visits": data["n_visits"][patient_idx],

        "observed_time": results["observed_time"][patient_idx],

        "predicted_time": results["predicted_time"][patient_idx],

        "event": results["event"][patient_idx],

    })

    # --------------------------------------------------
    # Tri chronologique des snapshots
    # --------------------------------------------------

    patient_predictions = patient_predictions.sort_values(

        "n_visits"

    ).reset_index(

        drop=True

    )

    return patient_predictions

In [ ]:
# ==========================================================
# APERÇU DES PRÉDICTIONS CHEZ LES PATIENTS AYANT ATTEINT L'IRT
# ==========================================================
#
# Affiche les premières prédictions réalisées chez les
# patients ayant présenté l'événement (IRT).
#
# Les colonnes présentées sont :
#   • patient_id      : identifiant du patient
#   • n_measures      : nombre de visites utilisées
#   • time_to_event   : temps restant observé avant l'IRT
#   • predicted_time  : temps restant prédit
#   • error           : erreur de prédiction
# ==========================================================

pred_cde.loc[

    pred_cde["event"] == 1,

    [

        "patient_id",

        "n_measures",

        "time_to_event",

        "predicted_time",

        "error",

    ],

].head(25)

In [ ]:
# ==========================================================
# CENSURES : T_PRÉDIT < T_OBSERVÉ
# ==========================================================

censored = pred_cde[
    pred_cde["event"] == 0
]

violations = censored[
    censored["predicted_time"] < censored["time_to_event"]
]

n_censored = len(censored)

n_violations = len(violations)

percentage = 100 * n_violations / n_censored

print(f"Patients censurés           : {n_censored}")
print(f"T_prédit < T_observé        : {n_violations}")
print(f"Pourcentage                 : {percentage:.2f}%")

In [ ]:
# ==========================================================
# HARRELL C-INDEX / UNO C-INDEX / MAE / RMSE
# PAR NOMBRE DE VISITES
# ==========================================================
#
# • Harrell et Uno : calculés sur tous les patients
#   (événements + censurés).
#
# • MAE et RMSE : calculés uniquement chez les patients
#   ayant atteint l'IRT.
# ==========================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from lifelines.utils import concordance_index

from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

import numpy as np
import pandas as pd


# --------------------------------------------------
# Données
# --------------------------------------------------

pred_df = pd.DataFrame({

    "n_visits": test_data["n_visits"],

    "observed_time": results_test["observed_time"],

    "predicted_time": results_test["predicted_time"],

    "event": results_test["event"].astype(bool),

})


# --------------------------------------------------
# Objet Surv pour Uno
# --------------------------------------------------

y_all = Surv.from_arrays(

    event=pred_df["event"],

    time=pred_df["observed_time"],

)


# --------------------------------------------------
# Calcul des métriques
# --------------------------------------------------

results_snapshot = []

for n_visits in sorted(pred_df["n_visits"].unique()):

    subset = pred_df.loc[
        pred_df["n_visits"] == n_visits
    ].copy()

    n_patients = len(subset)

    subset_events = subset.loc[
        subset["event"]
    ]

    n_events = len(subset_events)

    if n_events < 5:
        continue

    # ==========================================
    # Harrell C-index
    # ==========================================

    harrell = concordance_index(

        event_times=subset["observed_time"],

        predicted_scores=-subset["predicted_time"],

        event_observed=subset["event"],

    )

    # ==========================================
    # Uno C-index
    # ==========================================

    y_subset = Surv.from_arrays(

        event=subset["event"],

        time=subset["observed_time"],

    )

    try:

        uno = concordance_index_ipcw(

            y_all,          # jeu de référence

            y_subset,       # jeu évalué

            -subset["predicted_time"].values,

        )[0]

    except Exception:

        uno = np.nan

    # ==========================================
    # MAE
    # ==========================================

    mae = mean_absolute_error(

        subset_events["observed_time"],

        subset_events["predicted_time"],

    )

    # ==========================================
    # RMSE
    # ==========================================

    rmse = np.sqrt(

        mean_squared_error(

            subset_events["observed_time"],

            subset_events["predicted_time"],

        )

    )

    results_snapshot.append({

        "n_visits": int(n_visits),

        "patients": n_patients,

        "events": n_events,

        "harrell_c_index": harrell,

        "uno_c_index": uno,

        "mae": mae,

        "rmse": rmse,

    })


# --------------------------------------------------
# Tableau final
# --------------------------------------------------

results_snapshot = (
    pd.DataFrame(results_snapshot)
    .sort_values("n_visits")
    .reset_index(drop=True)
)

results_snapshot = results_snapshot.round({

    "harrell_c_index": 3,

    "uno_c_index": 3,

    "mae": 3,

    "rmse": 3,

})

results_snapshot

In [ ]:
# ==========================================================
# PLUS GRANDES SURESTIMATIONS DU TEMPS RESTANT AVANT L'IRT
# ==========================================================
#
# Affiche les patients pour lesquels le modèle surestime le
# plus le temps restant avant l'IRT.
#
# Une erreur positive correspond à :
#
#     predicted_time > time_to_event
#
# c'est-à-dire que le modèle prédit un temps restant plus
# long que celui réellement observé.
#
# Seuls les patients ayant atteint l'IRT sont considérés,
# puisque le temps réel jusqu'à l'événement est connu.
# ==========================================================

pred_cde.loc[

    pred_cde["event"] == 1,

].sort_values(

    by="error",

    ascending=False,

).loc[

    :,

    [

        "patient_id",

        "n_measures",

        "time_to_event",

        "predicted_time",

        "error",

        "absolute_error",

    ],

].head(20)

In [ ]:
# ==========================================================
# PLUS GRANDES SOUSESTIMATIONS DU TEMPS RESTANT AVANT L'IRT
# ==========================================================
#
# Affiche les patients pour lesquels le modèle sousestime le
# plus le temps restant avant l'IRT.
#
# Une erreur positive correspond à :
#
#     predicted_time < time_to_event
#
# c'est-à-dire que le modèle prédit un temps restant plus
# court que celui réellement observé.
#
# Seuls les patients ayant atteint l'IRT sont considérés,
# puisque le temps réel jusqu'à l'événement est connu.
# ==========================================================

pred_cde.loc[

    pred_cde["event"] == 1,

].sort_values(

    by="error",

    ascending=True,

).loc[

    :,

    [

        "patient_id",

        "n_measures",

        "time_to_event",

        "predicted_time",

        "error",

        "absolute_error",

    ],

].head(20)

### Visualisation MAE/RMSE par nombre de visites

In [ ]:
# ==========================================================
# MAE ET RMSE SELON LE NOMBRE DE VISITES
# ==========================================================
#
# Visualise l'évolution des performances du modèle en
# fonction du nombre de visites disponibles.
#
# Les étiquettes de l'axe des abscisses indiquent :
#
#     nombre de visites (nombre d'événements)
#
# Exemple :
#
#     4 (58)
#
# signifie :
#     • 4 visites disponibles
#     • 58 snapshots avec événement
# ==========================================================

import plotly.graph_objects as go

# ----------------------------------------------------------
# Noms des colonnes des métriques
# ----------------------------------------------------------

mae_col = "MAE" if "MAE" in results_snapshot.columns else "mae"

rmse_col = "RMSE" if "RMSE" in results_snapshot.columns else "rmse"

# ----------------------------------------------------------
# Étiquettes de l'axe des abscisses
# ----------------------------------------------------------

x_labels = [

    f"{n_visits} ({n_events})"

    for n_visits, n_events in zip(

        results_snapshot["n_visits"],

        results_snapshot["events"],

    )

]

fig = go.Figure()

# ----------------------------------------------------------
# Courbe du MAE
# ----------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_snapshot["n_visits"],

        y=results_snapshot[mae_col],

        mode="lines+markers",

        name="MAE",

        line=dict(

            width=3,

        ),

        marker=dict(

            size=8,

        ),

    )

)

# ----------------------------------------------------------
# Courbe du RMSE
# ----------------------------------------------------------

fig.add_trace(

    go.Scatter(

        x=results_snapshot["n_visits"],

        y=results_snapshot[rmse_col],

        mode="lines+markers",

        name="RMSE",

        line=dict(

            width=3,

        ),

        marker=dict(

            size=8,

        ),

    )

)

# ----------------------------------------------------------
# Mise en forme
# ----------------------------------------------------------

fig.update_xaxes(

    tickmode="array",

    tickvals=results_snapshot["n_visits"],

    ticktext=x_labels,

    title="Nombre de visites (nombre d'événements)",

)

fig.update_layout(

    title="Neural CDE-AFT — Performances selon le nombre de visites",

    xaxis_title="Nombre de visites",

    yaxis_title="Erreur (années)",

    template="plotly_white",

    width=900,

    height=550,

    hovermode="x unified",

    legend=dict(

        orientation="h",

        yanchor="bottom",

        y=1.02,

        xanchor="right",

        x=1,

    ),

)

fig.show()